# Notebook 02 — EDA Nâng cao
## Trực quan hóa nâng cao, phân tích tương quan, insights

**Mục tiêu:** Tìm ra các yếu tố quan trọng ảnh hưởng đến việc bỏ học.


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

from src.data.load_data import load_raw_data, TARGET_COL, FEATURE_COLS
from src.visualization.plots import (
    plot_categorical_vs_target, plot_correlation_heatmap,
    plot_top_corr_with_dropout
)

df = load_raw_data('../data/raw/dataset.csv')
PALETTE = {'Dropout': '#e74c3c', 'Enrolled': '#f39c12', 'Graduate': '#2ecc71'}
print('Dữ liệu đã tải ✅')

## 1. Tỷ lệ bỏ học theo các biến phân loại

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

cat_features = [
    ('Gender', {0: 'Nữ', 1: 'Nam'}),
    ('Scholarship holder', {0: 'Không học bổng', 1: 'Có học bổng'}),
    ('Debtor', {0: 'Không nợ', 1: 'Nợ học phí'}),
    ('Tuition fees up to date', {0: 'Chưa đóng HP', 1: 'Đã đóng HP'}),
    ('Displaced', {0: 'Tại địa phương', 1: 'Displaced'}),
    ('International', {0: 'Trong nước', 1: 'Quốc tế'}),
]

for i, (feat, label_map) in enumerate(cat_features):
    df_plot = df.copy()
    df_plot[feat] = df_plot[feat].map(label_map)
    sns.countplot(data=df_plot, x=feat, hue=TARGET_COL, palette=PALETTE, ax=axes[i], edgecolor='white')
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].legend(fontsize=9)

plt.suptitle('Tỷ lệ bỏ học theo các đặc trưng phân loại', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/02_categorical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Top đặc trưng tương quan với Dropout

In [ ]:
fig = plot_top_corr_with_dropout(df, top_n=15, save_path='../reports/figures/02_top_corr_dropout.png')
plt.show()

## 3. Heatmap tương quan

In [ ]:
# Encode target để đưa vào correlation
df_enc = df.copy()
df_enc[TARGET_COL] = df_enc[TARGET_COL].map({'Dropout': 0, 'Enrolled': 1, 'Graduate': 2})

# Lấy top 15 features tương quan với target
corr_with_target = df_enc.corr()[TARGET_COL].abs().sort_values(ascending=False)
top_cols = corr_with_target.head(16).index.tolist()

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones((len(top_cols), len(top_cols)), dtype=bool))
sns.heatmap(
    df_enc[top_cols].corr(),
    mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.3, annot_kws={'size': 8}
)
plt.title('Heatmap tương quan — Top 15 đặc trưng', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/02_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Phân tích điểm học tập theo nhãn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: điểm kỳ 1 vs kỳ 2
for label, color in PALETTE.items():
    subset = df[df[TARGET_COL] == label]
    axes[0].scatter(
        subset['Curricular units 1st sem (grade)'],
        subset['Curricular units 2nd sem (grade)'],
        c=color, label=label, alpha=0.4, s=15
    )
axes[0].set_xlabel('Điểm TB Kỳ 1', fontsize=11)
axes[0].set_ylabel('Điểm TB Kỳ 2', fontsize=11)
axes[0].set_title('Điểm kỳ 1 vs Kỳ 2 theo nhãn', fontsize=12, fontweight='bold')
axes[0].legend()

# Violin: điểm kỳ 1
sns.violinplot(data=df, x=TARGET_COL, y='Curricular units 1st sem (grade)',
               palette=PALETTE, ax=axes[1], inner='box')
axes[1].set_title('Phân phối điểm TB Kỳ 1 theo nhãn', fontsize=12, fontweight='bold')
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('../reports/figures/02_grade_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Tuổi nhập học và tỷ lệ bỏ học

In [ ]:
# Tính dropout rate theo nhóm tuổi
df['Age_group'] = pd.cut(df['Age at enrollment'],
                          bins=[16, 20, 23, 27, 35, 100],
                          labels=['17-20', '21-23', '24-27', '28-35', '35+'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar
age_target = df.groupby(['Age_group', TARGET_COL], observed=True).size().unstack(fill_value=0)
age_target_pct = age_target.div(age_target.sum(axis=1), axis=0)
age_target_pct[['Dropout', 'Enrolled', 'Graduate']].plot(
    kind='bar', stacked=True,
    color=[PALETTE['Dropout'], PALETTE['Enrolled'], PALETTE['Graduate']],
    ax=axes[0], edgecolor='white'
)
axes[0].set_title('Tỷ lệ bỏ học theo nhóm tuổi', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Nhóm tuổi')
axes[0].set_ylabel('Tỷ lệ')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(loc='upper right')

# Histogram tuổi
for label, color in PALETTE.items():
    subset = df[df[TARGET_COL] == label]['Age at enrollment']
    axes[1].hist(subset, bins=20, alpha=0.6, color=color, label=label, density=True)
axes[1].set_title('Phân phối tuổi nhập học theo nhãn', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Tuổi khi nhập học')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/02_age_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Xóa cột tạm
df.drop('Age_group', axis=1, inplace=True)

## 6. Tổng kết insights

In [ ]:
print('=== INSIGHTS TỪ EDA NÂNG CAO ===')
print()

# Dropout rate theo học bổng
scholarship_dropout = df.groupby('Scholarship holder')[TARGET_COL].apply(
    lambda x: (x == 'Dropout').mean() * 100
)
print('Tỷ lệ Dropout theo Học bổng:')
print(f'  Không học bổng: {scholarship_dropout[0]:.1f}%')
print(f'  Có học bổng   : {scholarship_dropout[1]:.1f}%')

# Dropout rate theo nợ học phí
debtor_dropout = df.groupby('Debtor')[TARGET_COL].apply(
    lambda x: (x == 'Dropout').mean() * 100
)
print('\nTỷ lệ Dropout theo Nợ học phí:')
print(f'  Không nợ: {debtor_dropout[0]:.1f}%')
print(f'  Có nợ   : {debtor_dropout[1]:.1f}%')

# Điểm TB theo nhãn
grade_by_label = df.groupby(TARGET_COL)['Curricular units 1st sem (grade)'].mean()
print('\nĐiểm TB Kỳ 1 theo nhãn:')
for label, grade in grade_by_label.items():
    print(f'  {label:12}: {grade:.2f}')

print()
print('KEY INSIGHTS:')
print('  1. Sinh viên có học bổng có tỷ lệ bỏ học thấp hơn đáng kể')
print('  2. Sinh viên nợ học phí có tỷ lệ bỏ học cao gấp 2-3 lần')
print('  3. Điểm kỳ 1 là tín hiệu mạnh nhất — Dropout có điểm thấp nhất')
print('  4. Sinh viên lớn tuổi (>27) có xu hướng bỏ học nhiều hơn')
print('  5. Số môn đạt kỳ 1 & kỳ 2 có tương quan dương mạnh với Graduate')